In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from concept_abstraction.training import *
from concept_abstraction.selection import *
from concept_abstraction.concept_bank import *
from concept_abstraction.env_utils import *
from concept_abstraction.environments import *
from concept_abstraction.utils import *
from concept_abstraction.environments import ConceptEnv

import sys 
import argparse
import secrets
import numpy as np 
import random 
import os
from stable_baselines3 import PPO
import pickle
import resource

In [3]:
is_jupyter = 'ipykernel' in sys.modules
is_main = __name__ == "__main__"

In [4]:
if is_main:
    if is_jupyter: 
        # Basics 
        seed        = 42
        environment_string = "mini_grid"
        training_timesteps = 250_000
        num_concepts_selected = 12
        selection_function = "q_value"
        # Experiment #1 & #2
        run_basic = False
        run_iterative = False
        run_two_stage = False 
        run_imperfect=False
        # Experiment #3
        cbm_accuracy_by_concept = [0.5 for i in range(24)]
        intervention_probability = 0.5
        intervention_accuracy_by_concept = [0.9 for i in range(24)] 
        cbm_std_by_concept = None 
        target_abstraction = 0.05
        reward_error = 0
        # Experiment #4
        concept_source = "human_selected_binary"
        # Experiment #5
        assess_completeness=False
        # Experiment #6
        num_iterations = 3
        selections_per_round = 1
        initial_concepts = 1
        out_folder = "imperfect"
    else:
        parser = argparse.ArgumentParser()
        parser.add_argument('--seed', help='Random Seed', type=int, default=42)
        parser.add_argument('--environment_string', help='Which environment to create', type=str, default="tree")
        parser.add_argument('--training_timesteps', help='Number of training timesteps', type=int, default=10000)
        parser.add_argument('--num_concepts_selected', help='Number of concepts selected by greedy or random',type=int, default=0)
        parser.add_argument('--selection_function', help='When selecting, use q_value, policy, or transition?', type=str, default="policy")
        parser.add_argument('--cbm_accuracy_by_concept', help="What is the accuracy of AI per concept?", nargs='*', type=float, default=None)
        parser.add_argument('--cbm_std_by_concept', help="What is the error of AI per concept?", nargs='*', type=float, default=None)
        parser.add_argument('--run_two_stage', help='Run the basic comparisons?', action='store_true')
        parser.add_argument('--run_iterative', help='Run the basic comparisons?', action='store_true')
        parser.add_argument('--run_basic', help='Run the basic comparisons?', action='store_true')
        parser.add_argument('--run_imperfect', help='Run the imperfect comparisons?', action='store_true')
        parser.add_argument('--target_abstraction', help='Value for the target abstraction with human performance', type=float, default=0.05)
        parser.add_argument('--reward_error', help="How much to perturb the reward by?", type=float, default=0)
        parser.add_argument('--concept_source', help='When selecting, use q_value, policy, or transition?', type=str, default="human_selected")
        parser.add_argument('--assess_completeness', help='Compare to the concept completeness algorithm?', action='store_true')
        parser.add_argument('--num_iterations', help='Number of iterations for iterative algorithms',type=int, default=0)
        parser.add_argument('--selections_per_round', help='Concepts to select per round',type=int, default=0)
        parser.add_argument('--initial_concepts', help='Number of starting/initial concepts',type=int, default=0)
        parser.add_argument('--out_folder', help='Which folder', type=str, default="exploration")

        args = parser.parse_args()

        seed = args.seed
        environment_string = args.environment_string
        training_timesteps = args.training_timesteps 
        num_concepts_selected = args.num_concepts_selected
        selection_function = args.selection_function
        cbm_accuracy_by_concept = args.cbm_accuracy_by_concept
        cbm_std_by_concept = args.cbm_std_by_concept
        run_basic = args.run_basic
        run_iterative = args.run_iterative
        run_two_stage = args.run_two_stage
        run_imperfect = args.run_imperfect
        target_abstraction = args.target_abstraction
        reward_error = args.reward_error
        concept_source = args.concept_source
        assess_completeness = args.assess_completeness
        num_iterations = args.num_iterations 
        selections_per_round = args.selections_per_round
        initial_concepts = args.initial_concepts
        out_folder = args.out_folder

    save_name = secrets.token_hex(4)  

In [5]:
if is_main:
        results = {}
        results['parameters'] = {'seed'      : seed,
                'environment_string'    : environment_string, 
                'training_timesteps': training_timesteps, 
                'selection_function': selection_function,
                'num_concepts_selected': num_concepts_selected,
                'cbm_accuracy_by_concept': cbm_accuracy_by_concept,
                'cbm_std_by_concept': cbm_std_by_concept,
                'intervention_probability': intervention_probability,
                'intervention_accuracy_by_concept': intervention_accuracy_by_concept,
                'target_abstraction': target_abstraction,
                'reward_error': reward_error, 
                'concept_source': concept_source,
                'assess_completeness': assess_completeness,
                'num_iterations': num_iterations,
                'selections_per_round': selections_per_round, 
                'initial_concepts': initial_concepts,
                'run_basic': run_basic,
                'run_iterative': run_iterative, 
                'run_two_stage': run_two_stage, 
        }
        print("Parameters {}".format(results['parameters']))

Parameters {'seed': 42, 'environment_string': 'mini_grid', 'training_timesteps': 250000, 'selection_function': 'q_value', 'num_concepts_selected': 12, 'cbm_accuracy_by_concept': [0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5], 'cbm_std_by_concept': None, 'intervention_probability': 0.5, 'intervention_accuracy_by_concept': [0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9], 'target_abstraction': 0.05, 'reward_error': 0, 'concept_source': 'human_selected_binary', 'assess_completeness': False, 'num_iterations': 3, 'selections_per_round': 1, 'initial_concepts': 1, 'run_basic': False, 'run_iterative': False, 'run_two_stage': False}


In [6]:
if is_main:
    np.random.seed(seed)
    random.seed(seed)

### Basic Setup

In [7]:
if is_main:
    concept_list = get_concepts(environment_string,concept_source,seed)
    num_concepts_selected = min(num_concepts_selected,len(concept_list))
    ground_truth_env, ground_truth_gym_env, additional_info = get_environment(environment_string, None, seed)   

In [8]:
if is_main:
    model_name = "../../results/models/env={}_training={}_seed={}.zip".format(environment_string,training_timesteps,seed)
    
    if os.path.exists(model_name):
        groundtruth_model = PPO.load(model_name)
    else:
        if "cyclic" in environment_string or "tree" in environment_string or "mimic" in environment_string:
            policy = "MlpPolicy"
        else:
            policy = "CnnPolicy"
        groundtruth_model = train_ppo_model(ground_truth_env,environment_string,total_timesteps=1,policy=policy)
        groundtruth_model.save(model_name)
    groundtruth_reward = evaluate_model(environment_string,ground_truth_gym_env,additional_info,groundtruth_model,seed)
    results['ground_truth'] = {'reward':groundtruth_reward}
    print(results['ground_truth']['reward'])

ValueError: Error: Unexpected observation shape (8, 1, 84, 84) for Box environment, please use (4, 84, 84) or (n_env, 4, 84, 84) for the observation shape.

### Basic Comparison

In [11]:
if is_main:
    
    model_name = "../../results/q_estimates/env={}_training={}_seed={}_selection={}_source={}.pkl".format(environment_string,training_timesteps,seed,selection_function,concept_source)
    results['basic_comparison'] = {}
    if os.path.exists(model_name):
        q_estimates = pickle.load(open(model_name,"rb"))
    else:
        if selection_function == "q_value":
            if environment_string == "mimic":
                modified_concepts = [lambda s, concept=concept: concept(additional_info['centers'][s]) 
                            for concept in concept_list]

                q_estimates = rollout_q_estimates_td(groundtruth_model,GymnasiumWrapper(DummyVecEnv([lambda: ground_truth_gym_env])),modified_concepts,learning_rate=1e-2,mimic=True,total_timesteps=5000,final_training=0)
            else:
                q_estimates = rollout_q_estimates_td(groundtruth_model,ground_truth_gym_env,concept_list)
        elif selection_function == "policy":
            if environment_string == "mimic":
                modified_concepts = [lambda s, concept=concept: concept(additional_info['centers'][s]) 
                            for concept in concept_list]

                q_estimates = rollout_pi_estimates(groundtruth_model,GymnasiumWrapper(DummyVecEnv([lambda: ground_truth_gym_env])),modified_concepts,mimic=True)
            else:
                q_estimates = rollout_pi_estimates(groundtruth_model,ground_truth_gym_env,concept_list)
        pickle.dump(q_estimates,open(model_name,"wb"))

In [12]:
if is_main and run_basic:
    # Train a random policy
    if environment_string == "mimic":
        model = RandomAgent(GymnasiumWrapper(DummyVecEnv([lambda: ground_truth_gym_env])))
    else:
        model = RandomAgent(ground_truth_gym_env)
    random_reward = evaluate_model(environment_string,ground_truth_gym_env,additional_info,model,seed)
    results['basic_comparison']['random'] = {'reward':random_reward}
    print(results['basic_comparison']['random']['reward'])

In [13]:
if is_main and run_basic:
    # Train a random selector
    subset_concept, random_idx = random_selection(concept_list,num_concepts_selected)
    subset_concept = [concept_list[i] for i in random_idx]
    env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
    model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy")
    random_selection_reward = evaluate_model(environment_string,eval_env,additional_info,model,seed)
    results['basic_comparison']['random_selection'] = {'reward':random_selection_reward, 'concepts': random_idx}
    print(results['basic_comparison']['random_selection']['reward'])

In [14]:
if True or is_main and run_basic:
    # Train a greedy selector
    subset_concept, greedy_idx = greedy_selection(concept_list,12,selection_function,q_estimates,concept_source)
    env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
    model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy")
    greedy_selection_reward = evaluate_model(environment_string,eval_env,additional_info,model,seed)
    results['basic_comparison']['greedy'] = {'concepts': greedy_idx, 'reward':greedy_selection_reward}
    print(results['basic_comparison']['greedy']['reward'])


Training: 250400it [02:02, 2036.63it/s]                            


256.9308510638298


In [15]:
if is_main and run_basic:
    subset_concept, greedy_iterative_idx = greedy_iterative_selection(concept_list,num_concepts_selected,selection_function,q_estimates,concept_source)
    env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
    model = train_ppo_model(env,environment_string,seed,total_timesteps=training_timesteps,policy="MlpPolicy")
    greedy_iterative_selection_reward = evaluate_model(environment_string,eval_env,additional_info,model,seed)
    results['basic_comparison']['greedy_iterative'] = {'concepts': greedy_iterative_idx, 'reward': greedy_iterative_selection_reward}
    print(results['basic_comparison']['greedy_iterative']['reward'])

In [16]:
if is_main and run_basic:
    subset_concept, lp_idx = lp_based_selection(ground_truth_gym_env,concept_list,num_concepts_selected,selection_function,q_estimates,concept_source)
    env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
    model = train_ppo_model(env,environment_string,total_timesteps=100_000,policy="MlpPolicy")
    lp_selection_reward = evaluate_model(environment_string,eval_env,additional_info,model,seed)
    results['basic_comparison']['lp'] = {'concepts': lp_idx, 'reward': lp_selection_reward}
    print(results['basic_comparison']['lp']['reward'])

### Imperfect Concept Predictors

In [71]:
if is_main and run_imperfect:
    results['inaccurate_comparison'] = {}


In [72]:
if is_main and run_imperfect:
    greedy_inaccurate_reward = {}
    for modification in ["continuous","binary"]:
        if modification == "continuous" and cbm_std_by_concept is not None:
            modified_concept_predictors = [inaccurate_concepts_continuous(func,0,std) for func,std in zip(concept_list,cbm_std_by_concept)]
        elif modification == "binary" and cbm_accuracy_by_concept is not None:
            modified_concept_predictors = [inaccurate_concepts_binary(func,acc,seed) for (func,acc) in zip(concept_list,cbm_accuracy_by_concept)]
        else:
            continue 
        _, greedy_idx = greedy_selection(concept_list,num_concepts_selected,selection_function,q_estimates,concept_source)
        subset_concept = [modified_concept_predictors[i] for i in greedy_idx]
        env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
        model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy")
        env, eval_env, additional_info = get_environment(environment_string,[concept_list[i] for i in greedy_idx],seed)

        greedy_inaccurate_reward[modification] = {
            'reward': evaluate_model(environment_string,eval_env,additional_info,model,seed),
            'concepts': greedy_idx
        }

    if greedy_inaccurate_reward != {}:
        results['inaccurate_comparison']['greedy'] = greedy_inaccurate_reward
        print(greedy_inaccurate_reward)

Training: 250400it [01:57, 2123.57it/s]                            


{'binary': {'reward': 9.347801683816652, 'concepts': [5, 7, 16, 18, 10, 4, 13, 22, 1, 12, 20, 8]}}


In [73]:
if is_main and run_imperfect:
    greedy_iterative_inaccurate_reward = {}
    for modification in ["continuous","binary"]:
        if modification == "continuous" and cbm_std_by_concept is not None:
            modified_concept_predictors = [inaccurate_concepts_continuous(func,0,std) for func,std in zip(concept_list,cbm_std_by_concept)]
        elif modification == "binary" and cbm_accuracy_by_concept is not None:
            modified_concept_predictors = [inaccurate_concepts_binary(func,acc,seed) for func,acc in zip(concept_list,cbm_accuracy_by_concept)]
        else:
            continue 
        
        _, greedy_iterative_idx = greedy_iterative_selection(concept_list,num_concepts_selected,selection_function,q_estimates,concept_source)
        subset_concept = [modified_concept_predictors[i] for i in greedy_iterative_idx]
        env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
        model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy")
        greedy_iterative_inaccurate_reward[modification] = {
            'reward': evaluate_model(environment_string,eval_env,additional_info,model,seed), 
            'concepts': greedy_iterative_idx
        } 

    if greedy_iterative_inaccurate_reward != {}:
        results['inaccurate_comparison']['greedy_iterative'] = greedy_iterative_inaccurate_reward
        print(greedy_iterative_inaccurate_reward)

Training: 250400it [01:54, 2177.98it/s]                            


{'binary': {'reward': 20.423187218353135, 'concepts': [5, 12, 10, 7, 18, 17, 1, 14, 23, 2, 20, 8]}}


In [74]:
if is_main and run_imperfect:
    lp_inaccurate_reward = {}
    for modification in ["continuous","binary"]:
        if modification == "continuous" and cbm_std_by_concept is not None:
            modified_concept_predictors = [inaccurate_concepts_continuous(func,0,std) for func,std in zip(concept_list,cbm_std_by_concept)]
        elif modification == "binary" and cbm_accuracy_by_concept is not None:
            modified_concept_predictors = [inaccurate_concepts_binary(func,acc,seed) for func,acc in zip(concept_list,cbm_accuracy_by_concept)]
        else:
            continue 
        _, lp_idx = lp_based_selection(ground_truth_gym_env,concept_list,num_concepts_selected,selection_function,q_estimates,concept_source)
        subset_concept = [modified_concept_predictors[i] for i in lp_idx]
        env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
        model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy")
        lp_inaccurate_reward[modification] = { 'reward': evaluate_model(environment_string,eval_env,additional_info,model,seed),
        'concepts': lp_idx}

    if lp_inaccurate_reward != {}:
        results['inaccurate_comparison']['lp'] = lp_inaccurate_reward
        print(lp_inaccurate_reward)

On 0 256 256
On 1 255 255
Set parameter Username
Academic license - for non-commercial use only - expires 2025-10-14


Training: 250400it [02:01, 2063.97it/s]                            


{'binary': {'reward': 15.296262254901961, 'concepts': [0, 2, 4, 6, 8, 11, 12, 15, 17, 19, 21, 22]}}


In [78]:
if is_main and run_imperfect:
    imperfect_lp_selection_reward = {}
    for modification in ["continuous","binary"]:
        if modification == "continuous" and cbm_std_by_concept is not None:
            modified_concept_predictors = [inaccurate_concepts_continuous(func,0,std) for func,std in zip(concept_list,cbm_std_by_concept)]
        elif modification == "binary" and cbm_accuracy_by_concept is not None:
            modified_concept_predictors = [inaccurate_concepts_binary(func,acc,seed) for func,acc in zip(concept_list,cbm_accuracy_by_concept)]
        else:
            continue 
        
        if modification == "continuous":
            subset_concept, imperfect_idx = imperfect_lp_selection(ground_truth_gym_env,modified_concept_predictors,groundtruth_model,selection_function,target_abstraction,num_concepts_selected,cbm_accuracy_by_concept,concept_source,environment_string,additional_info,direction='min')
        else:
            subset_concept, imperfect_idx = imperfect_lp_selection(ground_truth_gym_env,modified_concept_predictors,groundtruth_model,selection_function,target_abstraction,num_concepts_selected,cbm_accuracy_by_concept,concept_source,environment_string,additional_info,direction='max')
        env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
        model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy")
        imperfect_lp_selection_reward[modification] = evaluate_model(environment_string,eval_env,additional_info,model,seed)

    if imperfect_lp_selection_reward != {}:
        results['inaccurate_comparison']['imperfect_lp'] = imperfect_lp_selection_reward
        print(imperfect_lp_selection_reward)

Num 12
Starting stable training for sparse rewards...
Loss mean 0
Episode 25, Avg Reward: 19.08, Loss (mean/std/max): 1.23/0.04/1.26, Epsilon: 0.100
Episode 50, Avg Reward: 18.24, Loss (mean/std/max): 1.28/0.07/1.39, Epsilon: 0.100
Episode 75, Avg Reward: 20.40, Loss (mean/std/max): 1.27/0.07/1.39, Epsilon: 0.100
Episode 100, Avg Reward: 19.92, Loss (mean/std/max): 1.22/0.09/1.39, Epsilon: 0.099
Episode 125, Avg Reward: 27.44, Loss (mean/std/max): 1.19/0.10/1.39, Epsilon: 0.099
Episode 150, Avg Reward: 22.96, Loss (mean/std/max): 1.15/0.14/1.39, Epsilon: 0.099
Episode 175, Avg Reward: 18.92, Loss (mean/std/max): 1.11/0.16/1.39, Epsilon: 0.099
Episode 200, Avg Reward: 22.84, Loss (mean/std/max): 1.08/0.17/1.39, Epsilon: 0.099
Episode 225, Avg Reward: 28.20, Loss (mean/std/max): 1.04/0.18/1.39, Epsilon: 0.098
Episode 250, Avg Reward: 26.56, Loss (mean/std/max): 1.02/0.19/1.39, Epsilon: 0.098
Episode 275, Avg Reward: 19.40, Loss (mean/std/max): 1.00/0.19/1.39, Epsilon: 0.098
Episode 300, 

Training: 250400it [02:01, 2059.17it/s]                            


{'binary': 22.60887279311906}


### Intervention

In [66]:
if intervention_accuracy_by_concept is not None:
    results['intervention_comparison'] = {}

In [69]:
if is_main:
    greedy_intervention_reward = {}
    for modification in ["binary"]:
        if modification == "binary" and cbm_accuracy_by_concept is not None and intervention_accuracy_by_concept is not None:
            modified_concept_predictors = [inaccurate_concepts_binary_intervention(func,acc,intervene_acc,intervention_probability,seed) for (func,acc,intervene_acc) in zip(concept_list,cbm_accuracy_by_concept,intervention_accuracy_by_concept)]
        else:
            continue 
        _, greedy_idx = greedy_selection(concept_list,num_concepts_selected,selection_function,q_estimates,concept_source)
        subset_concept = [modified_concept_predictors[i] for i in greedy_idx]
        env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
        model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy")

        greedy_intervention_reward[modification] = {
            'reward': evaluate_model(environment_string,eval_env,additional_info,model,seed),
            'concepts': greedy_idx
        }

    if greedy_intervention_reward != {}:
        results['intervention_comparison']['greedy'] = greedy_intervention_reward
        print(greedy_intervention_reward)

Training: 250400it [02:00, 2078.44it/s]                            


{'binary': {'reward': 173.97173144876325, 'concepts': [5, 7, 16, 18, 10, 4, 13, 22, 1, 12, 20, 8]}}


In [79]:
if is_main:
    greedy_iterative_intervention_reward = {}
    for modification in ["binary"]:
        if modification == "binary" and cbm_accuracy_by_concept is not None and intervention_accuracy_by_concept is not None:
            modified_concept_predictors = [inaccurate_concepts_binary_intervention(func,acc,intervene_acc,intervention_probability,seed) for (func,acc,intervene_acc) in zip(concept_list,cbm_accuracy_by_concept,intervention_accuracy_by_concept)]
        else:
            continue 
        _, greedy_iterative_idx = greedy_iterative_selection(concept_list,num_concepts_selected,selection_function,q_estimates,concept_source)
        subset_concept = [modified_concept_predictors[i] for i in greedy_iterative_idx]
        env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
        model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy")

        greedy_iterative_intervention_reward[modification] = {
            'reward': evaluate_model(environment_string,eval_env,additional_info,model,seed),
            'concepts': greedy_iterative_idx
        }

    if greedy_iterative_intervention_reward != {}:
        results['intervention_comparison']['greedy_iterative'] = greedy_iterative_intervention_reward
        print(greedy_iterative_intervention_reward)

Training: 250400it [01:54, 2184.75it/s]                            


{'binary': {'reward': 184.82575757575756, 'concepts': [5, 12, 10, 7, 18, 17, 1, 14, 23, 2, 20, 8]}}


In [81]:
if is_main:
    lp_intervention_reward = {}
    for modification in ["binary"]:
        if modification == "binary" and cbm_accuracy_by_concept is not None and intervention_accuracy_by_concept is not None:
            modified_concept_predictors = [inaccurate_concepts_binary_intervention(func,acc,intervene_acc,intervention_probability,seed) for (func,acc,intervene_acc) in zip(concept_list,cbm_accuracy_by_concept,intervention_accuracy_by_concept)]
        else:
            continue 
        _, lp_idx = lp_based_selection(ground_truth_gym_env,concept_list,num_concepts_selected,selection_function,q_estimates,concept_source)
        subset_concept = [modified_concept_predictors[i] for i in lp_idx]
        env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
        model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy")

        lp_intervention_reward[modification] = {
            'reward': evaluate_model(environment_string,eval_env,additional_info,model,seed),
            'concepts': lp_idx
        }

    if lp_intervention_reward != {}:
        results['intervention_comparison']['lp'] = lp_intervention_reward
        print(lp_intervention_reward)

On 0 256 256
On 1 255 255


Training: 250400it [02:03, 2032.41it/s]                            


{'binary': {'reward': 177.36690647482015, 'concepts': [0, 2, 4, 6, 8, 11, 12, 15, 17, 19, 21, 22]}}


In [82]:
if is_main:
    imperfect_intervention_reward = {}
    for modification in ["binary"]:
        if modification == "binary" and cbm_accuracy_by_concept is not None and intervention_accuracy_by_concept is not None:
            modified_concept_predictors = [inaccurate_concepts_binary_intervention(func,acc,intervene_acc,intervention_probability,seed) for (func,acc,intervene_acc) in zip(concept_list,cbm_accuracy_by_concept,intervention_accuracy_by_concept)]
        else:
            continue 
        _, imperfect_idx = imperfect_lp_selection(ground_truth_gym_env,modified_concept_predictors,groundtruth_model,selection_function,target_abstraction,num_concepts_selected,intervention_accuracy_by_concept,concept_source,environment_string,additional_info,direction='max')
        subset_concept = [modified_concept_predictors[i] for i in imperfect_idx]
        env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
        model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy")

        imperfect_intervention_reward[modification] = {
            'reward': evaluate_model(environment_string,eval_env,additional_info,model,seed),
            'concepts': imperfect_idx
        }

    if imperfect_intervention_reward != {}:
        results['intervention_comparison']['imperfect_lp'] = imperfect_intervention_reward
        print(imperfect_intervention_reward)

Num 12
Starting stable training for sparse rewards...
Loss mean 0
Episode 25, Avg Reward: 18.68, Loss (mean/std/max): 1.25/0.03/1.29, Epsilon: 0.100
Episode 50, Avg Reward: 21.68, Loss (mean/std/max): 1.19/0.07/1.29, Epsilon: 0.100
Episode 75, Avg Reward: 24.80, Loss (mean/std/max): 1.18/0.08/1.29, Epsilon: 0.100
Episode 100, Avg Reward: 21.68, Loss (mean/std/max): 1.18/0.07/1.29, Epsilon: 0.099
Episode 125, Avg Reward: 24.76, Loss (mean/std/max): 1.16/0.07/1.29, Epsilon: 0.099
Episode 150, Avg Reward: 26.76, Loss (mean/std/max): 1.13/0.09/1.29, Epsilon: 0.099
Episode 175, Avg Reward: 22.40, Loss (mean/std/max): 1.11/0.10/1.29, Epsilon: 0.099
Episode 200, Avg Reward: 28.32, Loss (mean/std/max): 1.07/0.13/1.29, Epsilon: 0.099
Episode 225, Avg Reward: 23.20, Loss (mean/std/max): 1.05/0.14/1.29, Epsilon: 0.098
Episode 250, Avg Reward: 21.40, Loss (mean/std/max): 1.03/0.15/1.29, Epsilon: 0.098
Episode 275, Avg Reward: 21.52, Loss (mean/std/max): 1.00/0.17/1.29, Epsilon: 0.098
Episode 300, 

Training: 250400it [00:38, 6485.05it/s]                            


{'binary': {'reward': 206.31932773109244, 'concepts': [0, 5, 7, 8, 9, 10, 11, 14, 17, 20, 21, 22]}}


### Iterative

In [22]:
if is_main and run_iterative:
    env, eval_env, additional_info = get_environment(environment_string,concept_list,seed)
    gold_model = train_ppo_model(env,environment_string,total_timesteps=250_000,policy="MlpPolicy")
    results['iterative'] = {}

Training: 250400it [02:27, 1697.22it/s]                            


In [58]:
results['parameters']

{'seed': 42,
 'environment_string': 'cart_pole',
 'training_timesteps': 250000,
 'selection_function': 'q_value',
 'num_concepts_selected': 24,
 'cbm_accuracy_by_concept': None,
 'cbm_std_by_concept': None,
 'target_abstraction': 0.05,
 'reward_error': 0,
 'concept_source': 'human_selected_binary',
 'assess_completeness': False,
 'num_iterations': 3,
 'selections_per_round': 1,
 'initial_concepts': 1,
 'run_basic': False,
 'run_iterative': True,
 'run_two_stage': False}

In [23]:
if is_main and run_iterative:
    rand_idx = random_selection(concept_list,initial_concepts)[1]    
    rewards_iterative, concepts_iterative = iterative_selection(eval_env,gold_model,environment_string,rand_idx,concept_list,num_iterations,selections_per_round,seed)
    results['iterative']['iterative_selection'] = {'reward': rewards_iterative, 'concepts': concepts_iterative}
    print(rewards_iterative)

Training: 100%|██████████| 100000/100000 [00:36<00:00, 2727.15it/s]


On iteration 0
Steps 100 out of 10000
Steps 200 out of 10000
Steps 300 out of 10000
Steps 400 out of 10000
Steps 500 out of 10000
Steps 600 out of 10000
Steps 700 out of 10000
Steps 800 out of 10000
Steps 900 out of 10000
Steps 1000 out of 10000
Steps 1100 out of 10000
Steps 1200 out of 10000
Steps 1300 out of 10000
Steps 1400 out of 10000
Steps 1500 out of 10000
Steps 1600 out of 10000
Steps 1700 out of 10000
Steps 1800 out of 10000
Steps 1900 out of 10000
Steps 2000 out of 10000
Steps 2100 out of 10000
Steps 2200 out of 10000
Steps 2300 out of 10000
Steps 2400 out of 10000
Steps 2500 out of 10000
Steps 2600 out of 10000
Steps 2700 out of 10000
Steps 2800 out of 10000
Steps 2900 out of 10000
Steps 3000 out of 10000
Steps 3100 out of 10000
Steps 3200 out of 10000
Steps 3300 out of 10000
Steps 3400 out of 10000
Steps 3500 out of 10000
Steps 3600 out of 10000
Steps 3700 out of 10000
Steps 3800 out of 10000
Steps 3900 out of 10000
Steps 4000 out of 10000
Steps 4100 out of 10000
Steps 4200

Training: 100%|██████████| 100000/100000 [00:32<00:00, 3107.20it/s]


On iteration 1
Steps 100 out of 10000
Steps 200 out of 10000
Steps 300 out of 10000
Steps 400 out of 10000
Steps 500 out of 10000
Steps 600 out of 10000
Steps 700 out of 10000
Steps 800 out of 10000
Steps 900 out of 10000
Steps 1000 out of 10000
Steps 1100 out of 10000
Steps 1200 out of 10000
Steps 1300 out of 10000
Steps 1400 out of 10000
Steps 1500 out of 10000
Steps 1600 out of 10000
Steps 1700 out of 10000
Steps 1800 out of 10000
Steps 1900 out of 10000
Steps 2000 out of 10000
Steps 2100 out of 10000
Steps 2200 out of 10000
Steps 2300 out of 10000
Steps 2400 out of 10000
Steps 2500 out of 10000
Steps 2600 out of 10000
Steps 2700 out of 10000
Steps 2800 out of 10000
Steps 2900 out of 10000
Steps 3000 out of 10000
Steps 3100 out of 10000
Steps 3200 out of 10000
Steps 3300 out of 10000
Steps 3400 out of 10000
Steps 3500 out of 10000
Steps 3600 out of 10000
Steps 3700 out of 10000
Steps 3800 out of 10000
Steps 3900 out of 10000
Steps 4000 out of 10000
Steps 4100 out of 10000
Steps 4200

Training: 100%|██████████| 100000/100000 [00:51<00:00, 1944.15it/s]


On iteration 2
Steps 100 out of 10000
Steps 200 out of 10000
Steps 300 out of 10000
Steps 400 out of 10000
Steps 500 out of 10000
Steps 600 out of 10000
Steps 700 out of 10000
Steps 800 out of 10000
Steps 900 out of 10000
Steps 1000 out of 10000
Steps 1100 out of 10000
Steps 1200 out of 10000
Steps 1300 out of 10000
Steps 1400 out of 10000
Steps 1500 out of 10000
Steps 1600 out of 10000
Steps 1700 out of 10000
Steps 1800 out of 10000
Steps 1900 out of 10000
Steps 2000 out of 10000
Steps 2100 out of 10000
Steps 2200 out of 10000
Steps 2300 out of 10000
Steps 2400 out of 10000
Steps 2500 out of 10000
Steps 2600 out of 10000
Steps 2700 out of 10000
Steps 2800 out of 10000
Steps 2900 out of 10000
Steps 3000 out of 10000
Steps 3100 out of 10000
Steps 3200 out of 10000
Steps 3300 out of 10000
Steps 3400 out of 10000
Steps 3500 out of 10000
Steps 3600 out of 10000
Steps 3700 out of 10000
Steps 3800 out of 10000
Steps 3900 out of 10000
Steps 4000 out of 10000
Steps 4100 out of 10000
Steps 4200

Training: 100%|██████████| 100000/100000 [00:51<00:00, 1959.13it/s]


[184.22097378277152, 259.07936507936506, 254.64583333333334]


In [24]:
if is_main and run_iterative:
    td_learner = rollout_q_estimates_td(groundtruth_model,ground_truth_gym_env,concept_list,get_td_learner=True)
    rand_idx = random_selection(concept_list,initial_concepts)[1]    
    rewards_td, concepts_td = iterative_selection(eval_env,gold_model,environment_string,rand_idx,concept_list,num_iterations,selections_per_round,seed,td_learner=td_learner)
    results['iterative']['iterative_selection_q'] = {'reward': rewards_td, 'concepts': concepts_td}
    print(rewards_td)

Starting stable training for sparse rewards...
Loss mean 0


/usr0/home/naveenr/projects/concept_decisions/concept_abstraction/env_utils.py:370: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:274.)
  states = torch.FloatTensor(states)


Episode 25, Avg Reward: 20.40, Loss (mean/std/max): 1.23/0.02/1.24, Epsilon: 0.100
Episode 50, Avg Reward: 23.60, Loss (mean/std/max): 1.20/0.04/1.24, Epsilon: 0.100
Episode 75, Avg Reward: 28.12, Loss (mean/std/max): 1.16/0.06/1.24, Epsilon: 0.099
Episode 100, Avg Reward: 21.44, Loss (mean/std/max): 1.13/0.08/1.24, Epsilon: 0.099
Episode 125, Avg Reward: 21.84, Loss (mean/std/max): 1.11/0.08/1.24, Epsilon: 0.099
Episode 150, Avg Reward: 27.16, Loss (mean/std/max): 1.09/0.09/1.24, Epsilon: 0.099
Episode 175, Avg Reward: 18.32, Loss (mean/std/max): 1.07/0.10/1.24, Epsilon: 0.099
Episode 200, Avg Reward: 24.28, Loss (mean/std/max): 1.04/0.12/1.24, Epsilon: 0.099
Episode 225, Avg Reward: 21.76, Loss (mean/std/max): 1.02/0.13/1.24, Epsilon: 0.098
Episode 250, Avg Reward: 20.56, Loss (mean/std/max): 0.99/0.15/1.24, Epsilon: 0.098
Episode 275, Avg Reward: 20.64, Loss (mean/std/max): 0.97/0.16/1.24, Epsilon: 0.098
Episode 300, Avg Reward: 23.60, Loss (mean/std/max): 0.95/0.17/1.24, Epsilon: 0

Training: 100%|██████████| 100000/100000 [00:32<00:00, 3050.74it/s]


On iteration 0
Steps 100 out of 10000
Steps 200 out of 10000
Steps 300 out of 10000
Steps 400 out of 10000
Steps 500 out of 10000
Steps 600 out of 10000
Steps 700 out of 10000
Steps 800 out of 10000
Steps 900 out of 10000
Steps 1000 out of 10000
Steps 1100 out of 10000
Steps 1200 out of 10000
Steps 1300 out of 10000
Steps 1400 out of 10000
Steps 1500 out of 10000
Steps 1600 out of 10000
Steps 1700 out of 10000
Steps 1800 out of 10000
Steps 1900 out of 10000
Steps 2000 out of 10000
Steps 2100 out of 10000
Steps 2200 out of 10000
Steps 2300 out of 10000
Steps 2400 out of 10000
Steps 2500 out of 10000
Steps 2600 out of 10000
Steps 2700 out of 10000
Steps 2800 out of 10000
Steps 2900 out of 10000
Steps 3000 out of 10000
Steps 3100 out of 10000
Steps 3200 out of 10000
Steps 3300 out of 10000
Steps 3400 out of 10000
Steps 3500 out of 10000
Steps 3600 out of 10000
Steps 3700 out of 10000
Steps 3800 out of 10000
Steps 3900 out of 10000
Steps 4000 out of 10000
Steps 4100 out of 10000
Steps 4200

...

On iteration 1
Steps 100 out of 10000
Steps 200 out of 10000
Steps 300 out of 10000
Steps 400 out of 10000
Steps 500 out of 10000
Steps 600 out of 10000
Steps 700 out of 10000
Steps 800 out of 10000
Steps 900 out of 10000
Steps 1000 out of 10000
Steps 1100 out of 10000
Steps 1200 out of 10000
Steps 1300 out of 10000
Steps 1400 out of 10000
Steps 1500 out of 10000
Steps 1600 out of 10000
Steps 1700 out of 10000
Steps 1800 out of 10000
Steps 1900 out of 10000
Steps 2000 out of 10000
Steps 2100 out of 10000
Steps 2200 out of 10000
Steps 2300 out of 10000
Steps 2400 out of 10000
Steps 2500 out of 10000
Steps 2600 out of 10000
Steps 2700 out of 10000
Steps 2800 out of 10000
Steps 2900 out of 10000
Steps 3000 out of 10000
Steps 3100 out of 10000
Steps 3200 out of 10000
Steps 3300 out of 10000
Steps 3400 out of 10000
Steps 3500 out of 10000
Steps 3600 out of 10000
Steps 3700 out of 10000
Steps 3800 out of 10000
Steps 3900 out of 10000
Steps 4000 out of 10000
Steps 4100 out of 10000
Steps 4200

Training: 100%|██████████| 100000/100000 [00:35<00:00, 2804.95it/s]


On iteration 2
Steps 100 out of 10000
Steps 200 out of 10000
Steps 300 out of 10000
Steps 400 out of 10000
Steps 500 out of 10000
Steps 600 out of 10000
Steps 700 out of 10000
Steps 800 out of 10000
Steps 900 out of 10000
Steps 1000 out of 10000
Steps 1100 out of 10000
Steps 1200 out of 10000
Steps 1300 out of 10000
Steps 1400 out of 10000
Steps 1500 out of 10000
Steps 1600 out of 10000
Steps 1700 out of 10000
Steps 1800 out of 10000
Steps 1900 out of 10000
Steps 2000 out of 10000
Steps 2100 out of 10000
Steps 2200 out of 10000
Steps 2300 out of 10000
Steps 2400 out of 10000
Steps 2500 out of 10000
Steps 2600 out of 10000
Steps 2700 out of 10000
Steps 2800 out of 10000
Steps 2900 out of 10000
Steps 3000 out of 10000
Steps 3100 out of 10000
Steps 3200 out of 10000
Steps 3300 out of 10000
Steps 3400 out of 10000
Steps 3500 out of 10000
Steps 3600 out of 10000
Steps 3700 out of 10000
Steps 3800 out of 10000
Steps 3900 out of 10000
Steps 4000 out of 10000
Steps 4100 out of 10000
Steps 4200

Training: 100%|██████████| 100000/100000 [00:33<00:00, 3004.14it/s]


[42.13091216216216, 42.34664401019541, 42.082840236686394]


In [25]:
if is_main and run_iterative:
    num_concepts_selected = num_iterations*selections_per_round+initial_concepts
    bayesian_reward, bayesian_idx = bayesian_iterative_selection(ground_truth_gym_env,environment_string,seed,concept_list,num_iterations,num_concepts_selected)

    results['iterative']['bayesian'] = {
        'reward': bayesian_reward, 
        'concepts': bayesian_idx
    }
    print(bayesian_reward)

24
On iteration 0


Training: 100%|██████████| 100000/100000 [00:38<00:00, 2590.31it/s]


On iteration 1


Training: 100%|██████████| 100000/100000 [00:36<00:00, 2758.71it/s]


On iteration 2


Training: 100%|██████████| 100000/100000 [00:37<00:00, 2673.35it/s]


[260.620320855615, 253.25906735751295, 256.8578947368421, 261.75531914893617]


### Two-Stage Training

In [9]:
env, eval_env, additional_info = get_environment(environment_string,concept_list,seed)
gold_model = train_ppo_model(env,environment_string,total_timesteps=1_000_000,policy="MlpPolicy")
rew = evaluate_model(environment_string,eval_env,additional_info,gold_model,seed)
rew 

Training: 100%|██████████| 1000000/1000000 [04:40<00:00, 3570.30it/s]


0.963786231884058

In [11]:
concept_predictor = train_concept_predictor(ground_truth_gym_env,gold_model,concept_list)
two_stage_env, two_stage_gym_env, additional_info = get_environment(environment_string, concept_list, seed,fast_predictor=concept_predictor,use_processed=True)   
rew = evaluate_model(environment_string,two_stage_gym_env,additional_info,gold_model,seed)
rew 

📉 Epoch 1/20 | Train Loss 1.1571 | Val Loss 0.4241, Val Macro F1 0.8078 | Took 3.97 sec
📉 Epoch 2/20 | Train Loss 0.3545 | Val Loss 0.2965, Val Macro F1 0.8772 | Took 3.48 sec
📉 Epoch 3/20 | Train Loss 0.2688 | Val Loss 0.2913, Val Macro F1 0.8677 | Took 3.74 sec
📉 Epoch 4/20 | Train Loss 0.2465 | Val Loss 0.2214, Val Macro F1 0.9122 | Took 3.56 sec
📉 Epoch 5/20 | Train Loss 0.2184 | Val Loss 0.2094, Val Macro F1 0.9195 | Took 3.61 sec
📉 Epoch 6/20 | Train Loss 0.2121 | Val Loss 0.2029, Val Macro F1 0.9141 | Took 3.43 sec
📉 Epoch 7/20 | Train Loss 0.2012 | Val Loss 0.1934, Val Macro F1 0.9265 | Took 3.41 sec
📉 Epoch 8/20 | Train Loss 0.1972 | Val Loss 0.1918, Val Macro F1 0.9231 | Took 3.40 sec
📉 Epoch 9/20 | Train Loss 0.1906 | Val Loss 0.1858, Val Macro F1 0.9234 | Took 3.58 sec
📉 Epoch 10/20 | Train Loss 0.1942 | Val Loss 0.1828, Val Macro F1 0.9274 | Took 3.85 sec
📉 Epoch 11/20 | Train Loss 0.1826 | Val Loss 0.1796, Val Macro F1 0.9270 | Took 3.62 sec
📉 Epoch 12/20 | Train Loss 0.1

0.9637547762998792

#### Non-Experimental

In [ ]:
if is_main and run_two_stage:
    env, eval_env, additional_info = get_environment(environment_string,concept_list,seed)
    gold_model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy")

In [26]:
if is_main and not isinstance(ground_truth_env, ConceptEnv) and run_two_stage:
    greedy_two_stage = {}
    results['two_stage'] = {}
    greedy_concepts, greedy_idx = greedy_selection(concept_list,num_concepts_selected,selection_function,q_estimates,concept_source)

    concept_predictor = train_concept_predictor(ground_truth_gym_env,gold_model,concept_list)
    two_stage_env, two_stage_gym_env, additional_info = get_environment(environment_string, greedy_concepts, seed,fast_predictor=concept_predictor,use_processed=True)   
    model = train_ppo_model(two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=training_timesteps)    
    greedy_two_stage_reward = evaluate_model(environment_string,two_stage_gym_env,additional_info,model,seed)

    results['two_stage']['greedy'] = {'reward': greedy_two_stage_reward, 'concepts': greedy_idx}
    results['two_stage']['accuracy'] = acc_list 
    print(greedy_two_stage_reward)



In [27]:
if is_main and not isinstance(ground_truth_env, ConceptEnv) and run_two_stage:
    greedy_iterative_concepts, greedy_iterative_idx = greedy_iterative_selection(concept_list,num_concepts_selected,selection_function,q_estimates,concept_source)

    X,Y = get_concept_labels(ground_truth_gym_env,groundtruth_model,greedy_iterative_concepts)
    model = train_concept_predictor(X,Y)
    fast_predictor = FastGPUPredictor(model, "cuda")
    two_stage_env, two_stage_gym_env, additional_info = get_environment(environment_string, greedy_iterative_concepts, seed,fast_predictor=fast_predictor,use_processed=True)   
    model = train_ppo_model(two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=training_timesteps)    
    greedy_iterative_two_stage_reward = evaluate_model(environment_string,two_stage_gym_env,additional_info,model,seed)

    results['two_stage']['greedy_iterative'] = {'reward': greedy_iterative_two_stage_reward, 'concepts': greedy_iterative_idx}
    print(greedy_iterative_two_stage_reward)



In [28]:
if is_main and not isinstance(ground_truth_env, ConceptEnv) and run_two_stage:
    lp_concepts, lp_idx = lp_based_selection(ground_truth_gym_env,concept_list,num_concepts_selected,selection_function,q_estimates,concept_source)

    X,Y = get_concept_labels(ground_truth_gym_env,groundtruth_model,lp_concepts)
    model = train_concept_predictor(X,Y)
    fast_predictor = FastGPUPredictor(model, "cuda")
    two_stage_env, two_stage_gym_env, additional_info = get_environment(environment_string, lp_concepts, seed,fast_predictor=fast_predictor,use_processed=True)   
    model = train_ppo_model(two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=training_timesteps)    
    lp_two_stage_reward = evaluate_model(environment_string,two_stage_gym_env,additional_info,model,seed)

    results['two_stage']['lp'] = {'reward': lp_two_stage_reward, 'concepts': lp_idx}
    print(lp_two_stage_reward)



In [29]:
if is_main and not isinstance(ground_truth_env, ConceptEnv) and run_two_stage:
    acc_list = results['two_stage']['accuracy']
    top_k_idx = np.argsort(acc_list)[-num_concepts_selected:]
    top_k_concepts = [concept_list[i] for i in top_k_idx]
    
    X,Y = get_concept_labels(ground_truth_gym_env,groundtruth_model,top_k_concepts)
    model = train_concept_predictor(X,Y)
    fast_predictor = FastGPUPredictor(model, "cuda")
    two_stage_env, two_stage_gym_env, additional_info = get_environment(environment_string, top_k_concepts, seed,fast_predictor=fast_predictor,use_processed=True)   
    model = train_ppo_model(two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=training_timesteps)    
    top_k_two_stage_reward = evaluate_model(environment_string,two_stage_gym_env,additional_info,model,seed)

    results['two_stage']['top_k'] = {'reward': top_k_two_stage_reward, 'concepts': top_k_idx}
    print(top_k_two_stage_reward)



In [30]:
if is_main and not isinstance(ground_truth_env, ConceptEnv) and run_two_stage:
    acc_list = results['two_stage']['accuracy']
    imperfect_concepts, imperfect_idx = imperfect_lp_selection(ground_truth_gym_env,modified_concept_predictors,groundtruth_model,selection_function,target_abstraction,num_concepts_selected,acc_list,concept_source,environment_string,additional_info,direction='max')
    
    X,Y = get_concept_labels(ground_truth_gym_env,groundtruth_model,imperfect_concepts)
    model = train_concept_predictor(X,Y)
    fast_predictor = FastGPUPredictor(model, "cuda")
    two_stage_env, two_stage_gym_env, additional_info = get_environment(environment_string, imperfect_concepts, seed,fast_predictor=fast_predictor,use_processed=True)   
    model = train_ppo_model(two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=training_timesteps)    
    top_k_two_stage_reward = evaluate_model(environment_string,two_stage_gym_env,additional_info,model,seed)

    results['two_stage']['imperfect'] = {'reward': imperfect_two_stage_reward, 'concepts': imperfect_idx}
    print(imperfect_two_stage_reward)

## Ablations

### Reward Perturbation

In [31]:
if is_main and reward_error > 0:
    results['reward_error'] = {}
    perturbed_groundtruth_eval_env = RewardPerturbationWrapper(ground_truth_gym_env,reward_error)

    if selection_function == "q_value":
        if environment_string == "mimic":
            q_estimates_perturbed = rollout_q_estimates_td(groundtruth_model,GymnasiumWrapper(DummyVecEnv([lambda: ground_truth_gym_env])),concept_list,learning_rate=1e-3,mimic=True,total_timesteps=5000,final_training=0)
        else:
            q_estimates_perturbed = rollout_q_estimates_td(groundtruth_model,ground_truth_gym_env,concept_list)
    elif selection_function == "policy":
        if environment_string == "mimic":
            q_estimates_perturbed = rollout_pi_estimates(groundtruth_model,GymnasiumWrapper(DummyVecEnv([lambda: ground_truth_gym_env])),concept_list,mimic=True)
        else:
            q_estimates_perturbed = rollout_pi_estimates(groundtruth_model,ground_truth_gym_env,concept_list)

    subset_concept, greedy_perturbed_idx = greedy_selection(concept_list,num_concepts_selected,selection_function,q_estimates_perturbed,concept_source)
    env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
    perturbed_model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy")
    greedy_selection_perturbed_reward = evaluate_model(environment_string,eval_env,additional_info,perturbed_model,seed)
    results['reward_error']['greedy'] = {
        'reward': greedy_selection_perturbed_reward,
        'concepts': greedy_perturbed_idx
    }
    print(greedy_selection_perturbed_reward)

In [32]:
if is_main and reward_error > 0:
    subset_concept, greedy_perturbed_iterative_idx = greedy_iterative_selection(concept_list,num_concepts_selected,selection_function,q_estimates_perturbed,concept_source)
    env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
    model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy")
    greedy_iterative_selection_perturbed_reward = evaluate_model(environment_string,eval_env,additional_info,model,seed)
    results['reward_error']['greedy_iterative'] = {
        'reward': greedy_iterative_selection_perturbed_reward,
        'concepts': greedy_perturbed_iterative_idx
    }
    print(greedy_iterative_selection_perturbed_reward)

In [33]:
if is_main and reward_error > 0:
    subset_concept, lp_perturbed_idx = lp_based_selection(concept_list,num_concepts_selected,selection_function,q_estimates_perturbed,concept_source)
    env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
    model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy")
    lp_selection_perturbed_reward = evaluate_model(environment_string,eval_env,additional_info,model,seed)
    results['reward_error']['lp'] = {
        'reward': lp_selection_perturbed_reward,
        'concepts': lp_perturbed_idx
    }
    print(lp_selection_perturbed_reward)

### Comparison with Concept Completeness

In [34]:
# TODO: Create a Shapley-based baseline
if is_main and assess_completeness:
    pass 


## Save Data

In [99]:
save_path

'imperfect/e1c9fce8.json'

In [94]:
if is_main:
    save_path = get_save_path(out_folder,save_name)

In [95]:
if is_main:
    delete_duplicate_results(out_folder,"",results)

In [97]:
if is_main:
    json.dump(results,open('../../results/'+save_path,'w'))